# Project Template: Housing Price Prediction

- Module band: 00-03
- Estimated runtime: 10-15 minutes for the starter notebook
- Estimated project time: 4-5 hours
- Prerequisites: linear regression, regression metrics, train/validation/test splits
- Expected outputs: metric table, coefficient summary, residual plot, predicted-vs-actual plot, written interpretation


## Learning goals

- Build a reproducible regression workflow from data generation through evaluation.
- Compare a constant baseline against a linear model.
- Interpret coefficients and residuals in domain language.
- Write a short evidence-based conclusion about linear-model strengths and limits.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

SEED = 7
rng = np.random.default_rng(SEED)


def make_housing_dataset(sample_size: int = 320, seed: int = SEED):
    rng_local = np.random.default_rng(seed)
    square_feet = rng_local.normal(1850, 420, sample_size).clip(700, 3800)
    bedrooms = rng_local.integers(1, 6, sample_size)
    age_years = rng_local.uniform(0, 80, sample_size)
    school_score = rng_local.uniform(2.0, 10.0, sample_size)
    commute_score = rng_local.uniform(1.0, 10.0, sample_size)
    noise = rng_local.normal(0.0, 18000.0, sample_size)
    price = (
        65000
        + 145 * square_feet
        + 12000 * bedrooms
        - 950 * age_years
        + 9000 * school_score
        + 7000 * commute_score
        + noise
    )
    X = np.column_stack([square_feet, bedrooms, age_years, school_score, commute_score])
    feature_names = ["square_feet", "bedrooms", "age_years", "school_score", "commute_score"]
    return X, price, feature_names


X, y, feature_names = make_housing_dataset()
X.shape, y.shape


## Step 1: Frame the problem

Before running further analysis, add a short note here:

- Which features do you expect to raise price?
- Which features do you expect to lower price?
- Why is linear regression a reasonable baseline for this setting?


In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=SEED,
)

print({
    "train": X_train.shape,
    "val": X_val.shape,
    "test": X_test.shape,
})

for index, name in enumerate(feature_names):
    print(name, float(X[:, index].mean()), float(X[:, index].std()))


In [ ]:
# Baseline: predict the training-set mean
baseline_prediction = np.full_like(y_val, y_train.mean(), dtype=float)

# Main linear model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
val_prediction = linear_model.predict(X_val)

metrics = {
    "baseline_rmse": mean_squared_error(y_val, baseline_prediction) ** 0.5,
    "linear_rmse": mean_squared_error(y_val, val_prediction) ** 0.5,
    "linear_mae": mean_absolute_error(y_val, val_prediction),
    "linear_r2": r2_score(y_val, val_prediction),
}
metrics

# TODO:
# 1. Add one justified model variant.
# 2. Decide which model to carry to the test set.


In [ ]:
test_prediction = linear_model.predict(X_test)
residuals = y_test - test_prediction

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(test_prediction, y_test, alpha=0.7)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle="--", color="black")
axes[0].set_title("Predicted vs actual")
axes[0].set_xlabel("Predicted price")
axes[0].set_ylabel("Actual price")

axes[1].scatter(test_prediction, residuals, alpha=0.7)
axes[1].axhline(0.0, linestyle="--", color="black")
axes[1].set_title("Residual plot")
axes[1].set_xlabel("Predicted price")
axes[1].set_ylabel("Residual")
plt.tight_layout()

coefficient_table = list(zip(feature_names, linear_model.coef_))
coefficient_table


## Final analysis prompts

Answer these in complete sentences:

1. Which features appear most influential, and does that match your expectations?
2. Where does the model make its largest errors?
3. What do the residuals suggest about linearity and noise?
4. If you were restricted to Modules 00-03, what is the next improvement you would try?
